# LatentSync Lip-Sync for Living Codex™ Guides
Generates lip-synced videos for aoede, leda, selene, zephyr using LatentSync on Colab's free T4 GPU.

**Runtime → Change runtime type → T4 GPU** before running.

In [ ]:
# === ALL-IN-ONE FIX + RUN ===
# Fixes accelerate/peft version conflict, downloads videos, runs inference
import os, urllib.request

# 1. Fix the accelerate version conflict
!pip install accelerate>=0.34.0 peft --upgrade -q
print('✅ accelerate + peft upgraded')

# 2. Download idle videos + audio (Step 3 was never run!)
S3_BASE = 'https://justxempower-assets.s3.amazonaws.com/avatars/atlas'
AUDIO_URL = 'https://justxempower-assets.s3.amazonaws.com/avatars/atlas-script.mp3'
GUIDES = ['aoede', 'leda', 'selene', 'zephyr']

os.makedirs('/content/work', exist_ok=True)

audio_mp3 = '/content/work/atlas-script.mp3'
audio_wav = '/content/work/atlas-script.wav'
if not os.path.exists(audio_mp3):
    urllib.request.urlretrieve(AUDIO_URL, audio_mp3)
    print(f'✅ Audio downloaded')

if not os.path.exists(audio_wav):
    !ffmpeg -i {audio_mp3} -acodec pcm_s16le -ar 16000 -ac 1 {audio_wav} -y -loglevel error
    print(f'✅ Audio converted to WAV')

for guide in GUIDES:
    video_path = f'/content/work/{guide}-idle.mp4'
    if not os.path.exists(video_path):
        url = f'{S3_BASE}/{guide}/idle-video.mp4'
        urllib.request.urlretrieve(url, video_path)
        size_mb = os.path.getsize(video_path) / 1024 / 1024
        print(f'✅ {guide}: {size_mb:.1f}MB')
    else:
        print(f'⏩ {guide}: already downloaded')

# 3. Verify inputs
print('\n=== Input files ===')
for f in sorted(os.listdir('/content/work')):
    size = os.path.getsize(f'/content/work/{f}') / 1024 / 1024
    print(f'  {f} ({size:.1f}MB)')

# 4. Run LatentSync for all 4 guides
CONFIG = 'configs/unet/stage2.yaml'
CKPT = 'checkpoints/latentsync_unet.pt'

for guide in GUIDES:
    input_video = f'/content/work/{guide}-idle.mp4'
    output_video = f'/content/work/{guide}-lipsync.mp4'

    if os.path.exists(output_video):
        print(f'\n⏩ {guide}: already generated, skipping')
        continue

    print(f'\n{"─"*50}')
    print(f'🎬 {guide.upper()}')

    !cd /content/LatentSync && python -m scripts.inference \
        --unet_config_path {CONFIG} \
        --inference_ckpt_path {CKPT} \
        --video_path {input_video} \
        --audio_path /content/work/atlas-script.wav \
        --video_out_path {output_video} \
        --inference_steps 20 \
        --guidance_scale 2.0 \
        --seed 42

    if os.path.exists(output_video):
        size_mb = os.path.getsize(output_video) / 1024 / 1024
        print(f'✅ {guide} done ({size_mb:.1f}MB)')
    else:
        print(f'❌ {guide} FAILED')

# 5. Summary
print(f'\n{"═"*50}')
print('RESULTS:')
for guide in GUIDES:
    path = f'/content/work/{guide}-lipsync.mp4'
    if os.path.exists(path):
        print(f'  ✅ {guide}: {os.path.getsize(path)/1024/1024:.1f}MB')
    else:
        print(f'  ❌ {guide}: not generated')

In [ ]:
# Step 1: Clone LatentSync and install dependencies
!git clone https://github.com/bytedance/LatentSync.git /content/LatentSync 2>/dev/null || echo 'Already cloned'
%cd /content/LatentSync

# Fix pinned mediapipe version, then install
!sed -i 's/mediapipe==0.10.11/mediapipe>=0.10.13/' requirements.txt
!pip install -r requirements.txt -q
!pip install decord gdown -q
print('✅ Dependencies installed (including decord + mediapipe fix)')

In [ ]:
# Step 2: Download checkpoint (if setup_env.sh doesn't work, use HuggingFace)
%cd /content/LatentSync
!bash -c 'source setup_env.sh' || echo 'setup_env.sh failed, trying manual download...'

import os
ckpt_path = '/content/LatentSync/checkpoints/latentsync_unet.pt'
if not os.path.exists(ckpt_path):
    os.makedirs('/content/LatentSync/checkpoints', exist_ok=True)
    !pip install huggingface_hub -q
    from huggingface_hub import hf_hub_download
    hf_hub_download(repo_id='chunyu-li/LatentSync', filename='latentsync_unet.pt', local_dir='/content/LatentSync/checkpoints')
    print('✅ Checkpoint downloaded from HuggingFace')
else:
    print('✅ Checkpoint already exists')

!nvidia-smi | head -10

In [ ]:
# Step 4a: Diagnose — check what configs/checkpoints exist
import os, glob

print("=== Checkpoints ===")
for f in glob.glob('/content/LatentSync/checkpoints/**/*', recursive=True):
    size = os.path.getsize(f) / 1024 / 1024
    print(f'  {f} ({size:.1f}MB)')

print("\n=== Configs ===")
for f in glob.glob('/content/LatentSync/configs/**/*.yaml', recursive=True):
    print(f'  {f}')

print("\n=== Input files ===")
for f in glob.glob('/content/work/*'):
    size = os.path.getsize(f) / 1024 / 1024
    print(f'  {f} ({size:.1f}MB)')

print("\n=== GPU ===")
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
# Step 4b: Run LatentSync — one guide at a time with VISIBLE output
# Run this cell, then check the output. If a config is wrong, update the path below.
import os

# UPDATE THIS if the diagnostic cell shows a different config name
CONFIG = 'configs/unet/stage2.yaml'
CKPT = 'checkpoints/latentsync_unet.pt'

# Verify files exist
assert os.path.exists(f'/content/LatentSync/{CONFIG}'), f'Config not found: {CONFIG} — check diagnostic cell above for available configs'
assert os.path.exists(f'/content/LatentSync/{CKPT}'), f'Checkpoint not found: {CKPT} — check diagnostic cell above'

GUIDES = ['aoede', 'leda', 'selene', 'zephyr']

for guide in GUIDES:
    input_video = f'/content/work/{guide}-idle.mp4'
    output_video = f'/content/work/{guide}-lipsync.mp4'

    if os.path.exists(output_video):
        print(f'⏩ {guide}: already generated, skipping')
        continue

    print(f'\n{"─"*50}')
    print(f'🎬 {guide.upper()}')

    # Use ! so all stdout/stderr is visible in real-time
    !cd /content/LatentSync && python -m scripts.inference \
        --unet_config_path {CONFIG} \
        --inference_ckpt_path {CKPT} \
        --video_path {input_video} \
        --audio_path /content/work/atlas-script.wav \
        --video_out_path {output_video} \
        --inference_steps 20 \
        --guidance_scale 2.0 \
        --seed 42

    if os.path.exists(output_video):
        size_mb = os.path.getsize(output_video) / 1024 / 1024
        print(f'✅ {guide} done ({size_mb:.1f}MB)')
    else:
        print(f'❌ {guide} FAILED — check error above')

In [ ]:
# Step 5: Download lip-synced videos to your Mac
from google.colab import files
import os

for guide in ['aoede', 'leda', 'selene', 'zephyr']:
    path = f'/content/work/{guide}-lipsync.mp4'
    if os.path.exists(path):
        size_mb = os.path.getsize(path) / 1024 / 1024
        print(f'📥 Downloading {guide}-lipsync.mp4 ({size_mb:.1f}MB)...')
        files.download(path)
    else:
        print(f'⚠️ {guide}-lipsync.mp4 not found — re-run Step 4b')